In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any
import re
from tqdm import tqdm
import logging
import multiprocessing as mp
import time
import os

from camel.models import ModelFactory
from camel.types import ModelPlatformType
from camel.agents import ChatAgent

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class AIKeywordClassifier:
    """Uses the CAMEL framework"""
    
    def __init__(self, model_name: str = "llama3.1:8b"):
        """Initialize the classifier"""
        logger.info(f"Initializing AI keyword classifier, using model: {model_name}")
        
        # Initialize the CAMEL model
        self.model = ModelFactory.create(
            model_platform=ModelPlatformType.OLLAMA,
            model_type=model_name,
            url="http://localhost:11434/v1",
            model_config_dict={
                "temperature": 0.0,  # Fully deterministic
                "max_tokens": 400,  # Limit output length
            }
        )
        
        # Initialize the Agent
        self.agent = ChatAgent(
            system_message=self._get_system_prompt(),
            model=self.model,
            message_window_size=1
        )
        
        logger.info("AI keyword classifier initialization complete")
    
    def _get_system_prompt(self) -> str:
        """Get the system prompt"""
        return """You are a professional research expert in the AI / artificial intelligence field. Your task is to determine whether the given paper keywords belong to the AI field.
The AI field includes but is not limited to: Machine Learning, ML, Deep Learning, DL, Neural Networks, NN, Natural Language Processing, NLP, Computer Vision, CV, Knowledge Graph, KG, Recommendation Systems, Reinforcement Learning, RL, Expert Systems, Pattern Recognition, Data Mining, Speech Recognition, Image Processing, Intelligent Systems, Autonomous Driving, Robotics, Large Language Models, LLM, Transformer, BERT, GPT, Federated Learning, Graph Neural Networks, GNN
Please answer strictly in the following format:
- If the keywords belong to the AI field, answer: YES
- If the keywords do not belong to the AI field, answer: NO

Answer only YES or NO, without any other explanation."""

    def is_ai_related(self, keywords: List[str]) -> bool:
        """Determine whether the keyword list is AI-related"""
        if not keywords:
            return False
        
        # Build the query
        keywords_str = ", ".join(keywords)
        prompt = f"Determine whether the following keywords belong to the AI / artificial intelligence field:\n\nKeywords: {keywords_str}"
        
        try:
            # Get the model response
            response = self.agent.step(prompt)
            answer = response.msg.content.strip().upper()
            
            # Parse the response
            if "YES" in answer:
                return True
            elif "NO" in answer:
                return False
            else:
                # If the response format is non-standard, use the fallback rule
                logger.warning(f"Abnormal model response format: {answer}, using fallback rule for judgment")
                return self._fallback_classification(keywords)
                
        except Exception as e:
            logger.error(f"Model call failed: {str(e)}, using fallback rule for judgment")
            return self._fallback_classification(keywords)
    
    def _fallback_classification(self, keywords: List[str]) -> bool:
        """Fallback classification rule (based on keyword matching)"""
        ai_terms = {
            'machine learning', 'deep learning', 'neural network', 'artificial intelligence',
            'natural language processing', 'computer vision', 'nlp', 'ai', 'ml', 'dl',
            'cnn', 'rnn', 'lstm', 'gru', 'transformer', 'bert', 'gpt', 'llm',
            'reinforcement learning', 'supervised learning', 'unsupervised learning',
            'classification', 'regression', 'clustering', 'recommendation',
            'knowledge graph', 'expert system', 'intelligent', 'smart',
            'pattern recognition', 'feature extraction', 'data mining',
            'speech recognition', 'image recognition', 'object detection',
            'semantic', 'embedding', 'attention', 'gan', 'autoencoder',
            'federated learning', 'graph neural network', 'gnn'
        }
        
        # Check whether the keywords contain any AI term
        keywords_lower = [kw.lower() for kw in keywords]
        keywords_text = " ".join(keywords_lower)
        
        for ai_term in ai_terms:
            if ai_term in keywords_text:
                return True
        
        return False

# New multiprocessing worker function - processes data directly instead of files
def process_data_chunk_direct(data_chunk: List[dict], output_file: str, chunk_idx: int, 
                             batch_size: int = 1000, progress_interval: int = 100) -> int:
    """Process a data chunk directly, without creating temporary chunk files"""
    
    # Create the classifier
    classifier = AIKeywordClassifier()
    ai_papers_buffer = []  # Buffer for AI papers
    total_ai_count = 0
    processed_count = 0
    
    logger.info(f"Process {chunk_idx}: starting to process {len(data_chunk)} papers, writing every {batch_size} records")
    
    # Initialize the output file (clear it)
    with open(output_file, 'w', encoding='utf-8') as f:
        pass
    
    # Process papers
    with tqdm(total=len(data_chunk), desc=f"Process {chunk_idx}", position=chunk_idx) as pbar:
        for i, paper in enumerate(data_chunk):
            keywords = paper.get('keywords', [])
            
            # Determine whether it is an AI paper
            if classifier.is_ai_related(keywords):
                ai_papers_buffer.append(paper)
                total_ai_count += 1
            
            processed_count += 1
            
            # Write to file every time batch_size AI papers are buffered
            if len(ai_papers_buffer) >= batch_size:
                # Append the AI papers
                with open(output_file, 'a', encoding='utf-8') as f:
                    for ai_paper in ai_papers_buffer:
                        f.write(json.dumps(ai_paper, ensure_ascii=False) + '\n')
                
                logger.info(f"Process {chunk_idx}: wrote {len(ai_papers_buffer)} AI papers to file")
                ai_papers_buffer = []  # Clear the buffer
            
            # Update the progress bar every progress_interval papers
            if (i + 1) % progress_interval == 0 or (i + 1) == len(data_chunk):
                update_count = progress_interval if (i + 1) % progress_interval == 0 else (i + 1) % progress_interval
                pbar.update(update_count)
                pbar.set_postfix({
                    'AI papers': total_ai_count, 
                    'processed': processed_count,
                    'buffered': len(ai_papers_buffer)
                })
    
    # After processing all papers, write any remaining buffered data
    if ai_papers_buffer:
        with open(output_file, 'a', encoding='utf-8') as f:
            for ai_paper in ai_papers_buffer:
                f.write(json.dumps(ai_paper, ensure_ascii=False) + '\n')
        logger.info(f"Process {chunk_idx}: final write of remaining {len(ai_papers_buffer)} AI papers")
    
    logger.info(f"Process {chunk_idx}: done, found {total_ai_count} AI papers in total")
    return total_ai_count

class BatchStreamingWriter:
    """Streaming writer that supports batch writes"""
    
    def __init__(self, output_dir: Path, base_name: str, batch_size: int = 1000):
        self.output_dir = output_dir
        self.base_name = base_name
        self.batch_size = batch_size
        self.output_dir.mkdir(exist_ok=True)
        
        # File paths
        self.jsonl_file = self.output_dir / f"{base_name}.jsonl"
        self.csv_file = self.output_dir / f"{base_name}.csv"
        
        # Initialize files
        self._init_files()
        
        # Buffer and statistics
        self.papers_buffer = []
        self.written_count = 0
        
        logger.info(f"Batch writer initialization complete: {self.jsonl_file}, batch size: {batch_size}")
    
    def _init_files(self):
        """Initialize the output files"""
        # Clear the JSONL file
        with open(self.jsonl_file, 'w', encoding='utf-8') as f:
            pass
        
        # Create the CSV file and write the header
        csv_headers = [
            'id', 'title', 'abstract', 'year', 'venue', 'doc_type', 
            'n_citation', 'keywords', 'authors', 'doi', 'references_count'
        ]
        with open(self.csv_file, 'w', encoding='utf-8-sig', newline='') as f:
            import csv
            writer = csv.writer(f)
            writer.writerow(csv_headers)
    
    def add_paper(self, paper: dict):
        """Add a paper to the buffer"""
        self.papers_buffer.append(paper)
        
        # Write when the batch size is reached
        if len(self.papers_buffer) >= self.batch_size:
            self._flush_buffer()
    
    def _flush_buffer(self):
        """Write the buffered papers to file"""
        if not self.papers_buffer:
            return
        
        try:
            # Write JSONL
            with open(self.jsonl_file, 'a', encoding='utf-8') as f:
                for paper in self.papers_buffer:
                    f.write(json.dumps(paper, ensure_ascii=False) + '\n')
            
            # Write CSV
            with open(self.csv_file, 'a', encoding='utf-8-sig', newline='') as f:
                import csv
                writer = csv.writer(f)
                for paper in self.papers_buffer:
                    row_data = [
                        paper.get('id', ''),
                        paper.get('title', ''),
                        paper.get('abstract', ''),
                        paper.get('year', 0),
                        paper.get('venue', ''),
                        paper.get('doc_type', ''),
                        paper.get('n_citation', 0),
                        '; '.join(paper.get('keywords', [])),
                        '; '.join([auth.get('name', '') for auth in paper.get('authors', [])]),
                        paper.get('doi', ''),
                        len(paper.get('references', []))
                    ]
                    writer.writerow(row_data)
            
            self.written_count += len(self.papers_buffer)
            logger.info(f"Wrote {len(self.papers_buffer)} papers, total: {self.written_count}")
            
            # Clear the buffer
            self.papers_buffer = []
            
        except Exception as e:
            logger.error(f"Batch write failed: {e}")
    
    def finalize(self):
        """Finish writing, handling any remaining buffer"""
        if self.papers_buffer:
            logger.info(f"Writing remaining {len(self.papers_buffer)} papers")
            self._flush_buffer()
        
        logger.info(f"Batch writing complete, wrote {self.written_count} papers in total")
    
    def get_written_count(self) -> int:
        """Get the number of papers written"""
        return self.written_count

class DBLPAIExtractorBatch:
    """DBLP AI paper extractor that supports batch writes"""
    
    def __init__(self, data_file: str, output_dir: str = None):
        """Initialize the extractor"""
        self.data_file = Path(data_file)
        self.output_dir = Path(output_dir) if output_dir else self.data_file.parent
        self.output_dir.mkdir(exist_ok=True)
        
        logger.info(f"Data file: {self.data_file}")
        logger.info(f"Output directory: {self.output_dir}")
    
    def get_ai_keywords_extended(self) -> set:
        """Get the extended set of AI keywords used for pre-filtering"""
        return {
            # Core AI terms
            'artificial intelligence', 'machine learning', 'deep learning', 'neural network',
            'ai', 'ml', 'dl', 'nn', 'cnn', 'rnn', 'lstm', 'gru', 'transformer',
            
            # Specific AI techniques
            'natural language processing', 'computer vision', 'nlp', 'cv', 'bert', 'gpt',
            'reinforcement learning', 'supervised learning', 'unsupervised learning',
            'classification', 'regression', 'clustering', 'recommendation system',
            'knowledge graph', 'expert system', 'pattern recognition',
            'data mining', 'feature extraction', 'object detection',
            'speech recognition', 'image recognition', 'face recognition',
            'semantic analysis', 'sentiment analysis', 'text mining',
            'embedding', 'attention mechanism', 'gan', 'autoencoder',
            'federated learning', 'graph neural network', 'gnn',
            'random forest', 'svm', 'support vector machine', 'decision tree',
            'gradient boosting', 'xgboost', 'bayesian', 'markov',
            
            # AI application domains
            'intelligent system', 'smart system', 'autonomous', 'robotics',
            'chatbot', 'dialogue system', 'question answering',
            'information retrieval', 'search engine', 'recommendation',
            'personalization', 'collaborative filtering',
            
            # Emerging AI techniques
            'large language model', 'llm', 'foundation model', 'pretrained model',
            'few shot learning', 'zero shot learning', 'transfer learning',
            'meta learning', 'continual learning', 'multi task learning',
            'adversarial', 'explanation', 'interpretable ai', 'fairness',
            
            # Math / statistics related (closely related to AI)
            'optimization', 'gradient descent', 'backpropagation',
            'statistical learning', 'probabilistic model', 'monte carlo'
        }
    
    def quick_prefilter_ai_papers(self, year_threshold: int = 2010, chunk_size: int = 10000) -> List[dict]:
        """Quickly pre-filter AI papers"""
        
        ai_keywords = self.get_ai_keywords_extended()
        candidates = []
        total_processed = 0
        
        logger.info(f"Starting quick pre-filtering, chunk size: {chunk_size}")
        logger.info(f"Using {len(ai_keywords)} AI keywords for pre-filtering")
        
        start_time = time.time()
        
        with open(self.data_file, 'r', encoding='utf-8') as f:
            lines_buffer = []
            
            # First count the total number of lines
            logger.info("Counting total lines in file...")
            total_lines = sum(1 for _ in f)
            f.seek(0)
            logger.info(f"File contains {total_lines:,} records")
            
            # Use a progress bar
            with tqdm(total=total_lines, desc="Pre-filtering") as pbar:
                for line_num, line in enumerate(f, 1):
                    lines_buffer.append(line)
                    
                    # Process every chunk_size lines
                    if len(lines_buffer) >= chunk_size:
                        batch_candidates = self._process_batch_prefilter(
                            lines_buffer, ai_keywords, year_threshold
                        )
                        candidates.extend(batch_candidates)
                        
                        total_processed += len(lines_buffer)
                        
                        # Update the progress bar
                        pbar.update(len(lines_buffer))
                        pbar.set_postfix({
                            'candidates': len(candidates),
                            'processed': total_processed
                        })
                        
                        lines_buffer = []
                
                # Process remaining data
                if lines_buffer:
                    batch_candidates = self._process_batch_prefilter(
                        lines_buffer, ai_keywords, year_threshold
                    )
                    candidates.extend(batch_candidates)
                    total_processed += len(lines_buffer)
                    pbar.update(len(lines_buffer))
        
        elapsed_time = time.time() - start_time
        logger.info(f"Pre-filtering complete: {total_processed:,} -> {len(candidates)} candidates")
        logger.info(f"Pre-filtering took: {elapsed_time:.2f} seconds")
        logger.info(f"Filter rate: {(1 - len(candidates)/total_processed)*100:.2f}%")
        
        return candidates
    
    def _process_batch_prefilter(self, lines: List[str], ai_keywords: set, year_threshold: int) -> List[dict]:
        """Pre-filter a batch"""
        candidates = []
        
        for line in lines:
            try:
                paper = json.loads(line.strip())
                
                # Year filter
                year = paper.get('year', 0)
                if year < year_threshold:
                    continue
                
                # Keyword pre-filter
                keywords = paper.get('keywords', [])
                if not keywords:
                    continue
                    
                # Quick keyword matching
                keywords_text = ' '.join(keywords).lower()
                if any(ai_term in keywords_text for ai_term in ai_keywords):
                    candidates.append(paper)
                    
            except json.JSONDecodeError:
                continue
        
        return candidates
    
    def parallel_ai_classification_direct(self, candidates: List[dict], max_workers: int = None, 
                                        batch_size: int = 1000, progress_interval: int = 100) -> int:
        """Parallel AI classification that processes data directly, without creating temporary chunk files"""
        
        if max_workers is None:
            max_workers = min(mp.cpu_count() - 1, 4)
        
        logger.info(f"Starting direct parallel AI classification, using {max_workers} processes")
        logger.info(f"Number of candidate papers: {len(candidates)}")
        logger.info(f"Batch write size: {batch_size}")
        logger.info(f"Progress reporting interval: every {progress_interval} papers")
        
        # Create the temporary directory (used only for output files)
        temp_dir = self.output_dir / "temp_batch"
        temp_dir.mkdir(exist_ok=True)
        
        # Evenly distribute data across the processes
        chunk_size = len(candidates) // max_workers
        remainder = len(candidates) % max_workers
        
        data_chunks = []
        output_files = []
        start_idx = 0
        
        for i in range(max_workers):
            # Handle the remainder; the first few processes get one extra paper
            current_chunk_size = chunk_size + (1 if i < remainder else 0)
            end_idx = start_idx + current_chunk_size
            
            if start_idx >= len(candidates):
                break
                
            # Split the data directly, without saving to file
            chunk = candidates[start_idx:end_idx]
            output_file = temp_dir / f"output_{i}.jsonl"
            
            data_chunks.append(chunk)
            output_files.append(str(output_file))
            
            logger.info(f"Process {i}: assigned {len(chunk)} papers (index {start_idx}-{end_idx-1})")
            start_idx = end_idx
        
        # Process using a process pool
        start_time = time.time()
        
        with mp.Pool(processes=len(data_chunks)) as pool:
            # Submit tasks
            results = []
            for i, (data_chunk, output_file) in enumerate(zip(data_chunks, output_files)):
                result = pool.apply_async(process_data_chunk_direct, 
                                        (data_chunk, output_file, i, batch_size, progress_interval))
                results.append(result)
            
            # Wait for all tasks to complete
            total_ai_papers = 0
            with tqdm(total=len(results), desc="Parallel classification processes") as pbar:
                for result in results:
                    try:
                        ai_count = result.get(timeout=3600)  # 1-hour timeout
                        total_ai_papers += ai_count
                        pbar.update(1)
                        pbar.set_postfix({'AI papers': total_ai_papers})
                    except Exception as e:
                        logger.error(f"Process failed: {e}")
                        pbar.update(1)
        
        # Merge the result files
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        final_output = self.output_dir / f"ai_papers_{timestamp}.jsonl"
        final_csv = self.output_dir / f"ai_papers_{timestamp}.csv"
        
        # Merge the JSONL files
        logger.info("Starting to merge each process's output file...")
        with open(final_output, 'w', encoding='utf-8') as outf:
            for i, output_file in enumerate(output_files):
                if os.path.exists(output_file):
                    with open(output_file, 'r', encoding='utf-8') as inf:
                        content = inf.read()
                        if content.strip():  # Only write non-empty content
                            outf.write(content)
                            logger.info(f"Merged output file of process {i}")
                    os.remove(output_file)
        
        # Generate the CSV file
        self._convert_jsonl_to_csv(final_output, final_csv)
        
        # Clean up the temporary directory
        try:
            import shutil
            shutil.rmtree(temp_dir)
        except:
            pass
        
        elapsed_time = time.time() - start_time
        logger.info(f"Direct parallel classification complete: {len(candidates)} -> {total_ai_papers} AI papers")
        logger.info(f"Classification took: {elapsed_time:.2f} seconds")
        logger.info(f"Output file: {final_output}")
        
        return total_ai_papers
    
    def single_process_with_batch_write(self, candidates: List[dict], batch_size: int = 1000) -> int:
        """Single-process processing with batch writing"""
        
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        base_name = f"ai_papers_{timestamp}"
        
        # Create the batch writer
        writer = BatchStreamingWriter(self.output_dir, base_name, batch_size)
        
        # Create the classifier
        classifier = AIKeywordClassifier()
        ai_papers_count = 0
        
        logger.info(f"Starting single-process processing, batch size: {batch_size}")
        
        with tqdm(total=len(candidates), desc="AI classification") as pbar:
            for i, paper in enumerate(candidates):
                keywords = paper.get('keywords', [])
                if classifier.is_ai_related(keywords):
                    writer.add_paper(paper)
                    ai_papers_count += 1
                
                pbar.update(1)
                pbar.set_postfix({
                    'AI papers': ai_papers_count,
                    'buffered': len(writer.papers_buffer)
                })
        
        # Finish writing
        writer.finalize()
        
        return ai_papers_count
    
    def _convert_jsonl_to_csv(self, jsonl_file: Path, csv_file: Path):
        """Convert JSONL to CSV"""
        try:
            papers = []
            with open(jsonl_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        papers.append(json.loads(line.strip()))
            
            if papers:
                df_data = []
                for paper in papers:
                    row = {
                        'id': paper.get('id', ''),
                        'title': paper.get('title', ''),
                        'abstract': paper.get('abstract', ''),
                        'year': paper.get('year', 0),
                        'venue': paper.get('venue', ''),
                        'doc_type': paper.get('doc_type', ''),
                        'n_citation': paper.get('n_citation', 0),
                        'keywords': '; '.join(paper.get('keywords', [])),
                        'authors': '; '.join([auth.get('name', '') for auth in paper.get('authors', [])]),
                        'doi': paper.get('doi', ''),
                        'references_count': len(paper.get('references', []))
                    }
                    df_data.append(row)
                
                df = pd.DataFrame(df_data)
                df.to_csv(csv_file, index=False, encoding='utf-8-sig')
                logger.info(f"CSV file generated: {csv_file}")
        except Exception as e:
            logger.error(f"Failed to generate CSV: {e}")
    
    def extract_ai_papers_batch(self, year_threshold: int = 2010, max_workers: int = None, 
                               max_candidates: int = None, batch_size: int = 1000, 
                               progress_interval: int = 100) -> Dict[str, Any]:
        """AI paper extraction pipeline with batch writing (uses direct data processing)"""
        total_start_time = time.time()
        
        # Step 1: quick pre-filtering
        logger.info("Step 1: quick pre-filtering...")
        candidates = self.quick_prefilter_ai_papers(year_threshold)
        
        if not candidates:
            logger.warning("No candidate papers after pre-filtering")
            return {'ai_papers_count': 0}
        
        logger.info(f"Pre-filtering yielded {len(candidates)} candidate papers")
        
        # Limit the number of candidates
        if max_candidates and len(candidates) > max_candidates:
            logger.info(f"Too many candidates ({len(candidates)}), limiting to {max_candidates}")
            candidates = candidates[:max_candidates]
        
        # Step 2: AI classification (uses direct data processing)
        logger.info("Step 2: precise AI model classification (direct data processing)...")
        
        if len(candidates) > 5000 and max_workers and max_workers > 1:  # Use parallel processing
            ai_papers_count = self.parallel_ai_classification_direct(
                candidates, max_workers, batch_size, progress_interval)
        else:
            # Single-process processing
            ai_papers_count = self.single_process_with_batch_write(candidates, batch_size)
        
        total_elapsed = time.time() - total_start_time
        logger.info(f"Full pipeline took: {total_elapsed:.2f} seconds")
        logger.info(f"Final result: {ai_papers_count} AI papers")
        
        return {
            'ai_papers_count': ai_papers_count,
            'processing_time': total_elapsed
        }

def main():
    """Main function"""
    print("DBLP AI Paper Extractor (direct data processing version)")
    print("=" * 50)
    
    # Configuration parameters
    data_file = "Data/DBLP-Citation-network-V18.jsonl"
    output_dir = "Data/DBLP_AI"
    year_threshold = 2010
    
    # Check the data file
    if not Path(data_file).exists():
        print(f"Data file does not exist: {data_file}")
        return
    
    try:
        # Initialize the extractor
        extractor = DBLPAIExtractorBatch(data_file, output_dir)
        
        # User confirmation
        print(f"Data file: {data_file}")
        print(f"Output directory: {output_dir}")
        print(f"Year threshold: {year_threshold}")
        
        response = input("\nStart extraction? (y/N): ").lower().strip()
        if response != 'y':
            print("Extraction cancelled")
            return
        
        # Ask for the number of parallel processes
        max_workers = input(f"Number of parallel processes (default auto, max {mp.cpu_count()-1}): ").strip()
        max_workers = int(max_workers) if max_workers.isdigit() else None
        
        # Ask whether to limit the number of candidates
        max_candidates = input("Limit number of candidates (default unlimited): ").strip()
        max_candidates = int(max_candidates) if max_candidates.isdigit() else None
        
        # Ask for the batch write size
        batch_size = input("Batch write size (default 1000): ").strip()
        batch_size = int(batch_size) if batch_size.isdigit() else 1000
        
        # Ask for the progress reporting interval
        progress_interval = input("Progress reporting interval (default 100 papers): ").strip()
        progress_interval = int(progress_interval) if progress_interval.isdigit() else 100
        
        print(f"\nStarting direct data processing pipeline (batch size: {batch_size}, progress interval: {progress_interval})...")
        
        # Start extraction
        results = extractor.extract_ai_papers_batch(
            year_threshold=year_threshold,
            max_workers=max_workers,
            max_candidates=max_candidates,
            batch_size=batch_size,
            progress_interval=progress_interval
        )
        
        # Show the results
        if results['ai_papers_count'] > 0:
            print(f"\nExtraction complete, found {results['ai_papers_count']} AI-related papers")
            print(f"Total time: {results['processing_time']:.2f} seconds")
            print(f"Output files saved in: {output_dir}")
            print(f"Batch writing: writes every {batch_size} records to reduce memory usage")
            print(f"Progress reporting: reports every {progress_interval} papers")
        else:
            print("\nNo AI papers matching the criteria were found")
            
    except KeyboardInterrupt:
        print("\nUser interrupted extraction")
        print("Already-processed data has been saved via batch writing")
    except Exception as e:
        print(f"\nExtraction failed: {str(e)}")
        logger.exception("Detailed error information:")

if __name__ == "__main__":
    main()

In [ ]:
import json
import pandas as pd

# Preview the AI papers data file
def preview_ai_papers():
    file_path = 'Data/DBLP_AI/ai_papers_20250831_215837.jsonl'
    
    try:
        print(f"Previewing file: {file_path}")
        
        # Read the first few lines of data
        papers = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= 5:  # Only read the first 5 lines
                    break
                papers.append(json.loads(line.strip()))
        
        print(f"\nPreview of the first 5 records in the file:")
        print("-" * 80)
        
        for i, paper in enumerate(papers, 1):
            print(f"\nRecord {i}:")  
            print(json.dumps(paper, indent=2, ensure_ascii=False))
            print("-" * 80)
            
        # Get the total number of lines in the file
        with open(file_path, 'r', encoding='utf-8') as f:
            total_lines = sum(1 for _ in f)
        print(f"\nThe file contains {total_lines} AI papers in total")
            
    except FileNotFoundError:
        print(f"Error: file {file_path} does not exist")
    except json.JSONDecodeError as e:
        print(f"Error: JSON parsing failed - {e}")
    except Exception as e:
        print(f"Error: {e}")

# Run the preview
preview_ai_papers()
